In [0]:
# %python
# =========================================
# SILVER — Meta de Alfabetização Brasil
# Fonte: Bronze meta_alfabetizacao_brasil.parquet (sempre o mais recente)
# Saída: silver.meta_brasil (Delta, particionado por ano)
# Chave composta: ano + rede
# =========================================
import io, sys
from pathlib import Path

repo = Path.cwd()
while not repo.name.startswith("postech-aisc") and repo.parent != repo:
    repo = repo.parent
sys.path.insert(0, str(repo))

from src.config.settings import AZURE_STORAGE_ACCOUNT, AZURE_STORAGE_KEY, BRONZE_CONTAINER
from azure.storage.blob import BlobServiceClient
import pandas as pd

# ---------- 1. LER DO BRONZE (SEMPRE O MAIS RECENTE) ----------
conn_str = (f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};"
            f"AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net")
blob_service_client = BlobServiceClient.from_connection_string(conn_str)
container_client = blob_service_client.get_container_client(BRONZE_CONTAINER)

prefixo = "meta_alfabetizacao_brasil.parquet"
candidatos = [b.name for b in container_client.list_blobs() if b.name.endswith(prefixo)]
if not candidatos:
    raise ValueError(f"Nenhum arquivo encontrado com prefixo: {prefixo}")
arquivo_mais_recente = sorted(candidatos)[-1]  # YYYY-MM-DD no início = mais recente
print(f"Lendo arquivo mais recente: {arquivo_mais_recente}")

data = container_client.get_blob_client(arquivo_mais_recente).download_blob().readall()
pdf = pd.read_parquet(io.BytesIO(data))

# ---------- 2. TRANSFORMAÇÕES ----------
# Padronizar chaves
pdf["ano"] = pdf["ano"].astype(str).str.strip()
pdf["rede"] = pdf["rede"].astype(str).str.strip()
# Mapear rede (2=Estadual, 3=Municipal, 4=Privada) — confirmar no dicionário
rede_map = {"2": "Estadual", "3": "Municipal", "4": "Privada"}
pdf["rede_nome"] = pdf["rede"].map(rede_map).fillna("Desconhecida")
# Metadados de rastreabilidade (princípio da arquitetura)
pdf["ingested_at"] = pdf["_ingested_at"]
pdf["source"] = f"bronze/{arquivo_mais_recente}"
pdf["version"] = "1.0"

# ---------- 3. DATA QUALITY ----------
chaves = ["ano", "rede"]
nulos_chave = pdf[chaves].isna().sum().sum()
dups = pdf.duplicated(subset=chaves).sum()
print(f"[DQ] Nulos nas chaves: {nulos_chave}")
print(f"[DQ] Duplicados na chave composta: {dups}")

# ---------- 3.5 PERSISTIR DQ NO MONITORAMENTO ----------
spark.sql("CREATE DATABASE IF NOT EXISTS monitoring")
spark.sql("""
  CREATE TABLE IF NOT EXISTS monitoring.dq_results (
    table_name STRING, rule STRING, status STRING,
    records_checked BIGINT, failures BIGINT, run_at TIMESTAMP
  ) USING DELTA
""")

from pyspark.sql import functions as F

registros = [
    ("silver.meta_brasil", "completude_chaves",
     "PASS" if nulos_chave == 0 else "FAIL", int(len(pdf)), int(nulos_chave)),
    ("silver.meta_brasil", "unicidade_chave_composta",
     "PASS" if dups == 0 else "FAIL", int(len(pdf)), int(dups)),
]
df_dq = spark.createDataFrame(registros,
    ["table_name", "rule", "status", "records_checked", "failures"]) \
    .withColumn("run_at", F.current_timestamp())
df_dq.write.mode("append").saveAsTable("monitoring.dq_results")

# ---------- 4. GRAVAR EM DELTA ----------
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
df = spark.createDataFrame(pdf)
df = df.drop("_ingested_at", "_source_table")

# overwriteSchema: recria a tabela com o novo esquema (resolve erro de merge do 'ano')
df.write.mode("overwrite").option("overwriteSchema", "true") \
    .format("delta").partitionBy("ano") \
    .saveAsTable("silver.meta_brasil")

print(f"\n[OK] Silver meta_brasil gravada | Registros: {df.count()} | Partições: {df.select('ano').distinct().count()}")

In [0]:
%sql
-- Quais UFs estão presentes na silver
SELECT sigla_uf, COUNT(*) AS registros
FROM silver.indicador_uf
GROUP BY sigla_uf
ORDER BY sigla_uf;

In [0]:
%sql
-- UFs oficiais do Brasil (26 estados + DF)
WITH ufs_oficiais AS (
  SELECT explode(array(
    'AC','AL','AP','AM','BA','CE','DF','ES','GO',
    'MA','MT','MS','MG','PA','PB','PR','PE','PI',
    'RJ','RN','RS','RO','RR','SC','SP','SE','TO'
  )) AS sigla_uf
)
SELECT
  u.sigla_uf,
  CASE WHEN s.sigla_uf IS NULL THEN 'FALTANDO' ELSE 'OK' END AS status
FROM ufs_oficiais u
LEFT JOIN (SELECT DISTINCT sigla_uf FROM silver.indicador_uf) s
  ON u.sigla_uf = s.sigla_uf
ORDER BY u.sigla_uf;

In [0]:
# %python
import io, pandas as pd

# Lê o mesmo arquivo do Bronze que o notebook 2 usa
data = container_client.get_blob_client("2026-08-30_uf.parquet").download_blob().readall()
pdf_bronze = pd.read_parquet(io.BytesIO(data))

# Procura RR e DF na fonte
filtro = pdf_bronze["sigla_uf"].astype(str).str.strip().str.upper().isin(["RR", "DF"])
print(f"Registros de RR/DF no Bronze: {filtro.sum()}")

# UFs presentes na fonte
print("UFs no Bronze:")
print(sorted(pdf_bronze["sigla_uf"].astype(str).str.strip().str.upper().unique()))